# 02 | Train a facial-hyperpigmentation LoRA with SD2.1

Use **only real, group-separated hyperpigmentation TRAIN images**. The official Diffusers v0.35.1 trainer updates UNet LoRA weights while leaving the pretrained SD2.1 base frozen.

**No subtype labels?** General hyperpigmentation captions are acceptable for a broad LoRA, but they do **not** make the model reliable for melasma, PIH, and solar lentigines individually. Supply clinician-reviewed caption CSVs for subtype control. Never invent a diagnosis from the image.

The pretrained `hcarrion/hyperpigmentation` and `hcarrion/solar_lentigo` textual inversions are **NOT used to initialize LoRA training**. This deliberately keeps the training notebook on the tested official upstream path. You can combine them with your saved LoRA **during inference** in Notebook 01.

**Local verification** covers Python compilation, synthetic image-folder fixtures and command structure, not GPU training. Training uses FP16 on T4 and starts with 20 steps as a smoke test.

**Audit enhancement:** source images are orientation-corrected, transparency-composited and aspect-ratio-padded to preserve facial morphology; the generated LoRA includes a backbone provenance record.

## 1. Packages
Enable a Kaggle T4 GPU and Internet. Do not reinstall Kaggle's GPU-enabled torch. If the notebook has stale imports, restart once after installing packages.

In [ ]:
import sys, subprocess
PACKAGES = [
    'diffusers==0.35.1', 'transformers==4.56.2',
    'accelerate==1.10.1', 'peft==0.17.1',
    'datasets==3.6.0', 'huggingface_hub==0.34.4',
    'safetensors==0.6.2', 'ftfy==6.3.1',
    'imagehash==4.3.2', 'bitsandbytes==0.47.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
print('Pinned dependencies installed. Restart this session if imports had already been loaded.')

In [ ]:
import json, csv, hashlib, shutil, urllib.request, sys, subprocess, random
from pathlib import Path
from collections import Counter
import numpy as np
import imagehash
from PIL import Image, ImageOps
from sklearn.model_selection import GroupShuffleSplit
import torch, diffusers, transformers, accelerate, datasets, peft
from IPython.display import display
from huggingface_hub import model_info
assert torch.cuda.is_available(), 'Enable a Kaggle GPU (T4 recommended) in notebook settings.'
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.get_device_name(0))
print('Diffusers', diffusers.__version__, '| Datasets', datasets.__version__)

## 2. Approved Kaggle real-image dataset v2
The sole raw real-image source is `/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection`. Original images in that source are **not directly passed into LoRA training**. Instead, attach the pre-reviewed, deduplicated frozen real train manifest, split images, and a passed real-label audit. Train only approved hyperpigmentation examples. The notebook stages only train examples and streams trainer logs to Kaggle outputs. The optional run may fail if checkpoint/API dependencies or source permissions are unavailable.


**Updated split protocol:** this notebook explicitly reports the selected class's train/validation/test counts from the already reviewed GLOBAL image-group split, saves `class_frozen_split_counts.csv` and the private assignment CSV, and trains LoRA **only** on that class's training images. It will not create an independent raw-folder split that could leak repeated patients across classes. Complete console output is in `notebook_console.log` and the actual trainer log.


In [ ]:
IMAGE_ROOT = Path('/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection/Face_Skin_dataset/Hyper_Pigmentation')  # REQUIRED EDIT
SPLIT_CSV = ''      # optional: CSV with filename,split
CAPTIONS_CSV = ''   # optional: CSV with filename,caption
AUTO_SPLIT = False  # fallback provisional 70/15/15 split if no train/ or split CSV
BASE_MODEL = 'sd2-community/stable-diffusion-2-1-base'
OUT_DIR = Path('/kaggle/working/hyperpigmentation_lora_output')
DATA_PREP = Path('/kaggle/working/hyperpigmentation_lora_training_data')
STEPS = 20         # GPU smoke test; set to 1000 only after successful first test
RANK = 16
RESOLUTION = 512
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 1e-4
SEED = 42031
CHECKPOINT_STEPS = 250
USE_8BIT_ADAM = True
assert STEPS > 0 and RESOLUTION % 8 == 0 and RANK > 0
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f'Set IMAGE_ROOT to an existing Kaggle input folder; got {IMAGE_ROOT}')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PREP.mkdir(parents=True, exist_ok=True)

In [ ]:
# Only Kaggle v2 real images reviewed and assigned to the FROZEN real TRAIN split.
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prepare_lora_train_only,append_run_log
RAW_REAL_DATASET_V2='/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection'
REAL_TRAIN_MANIFEST='/kaggle/input/EDIT_APPROVED_REAL_SPLIT/real_all_manifest.csv'
REAL_TRAIN_IMAGE_ROOT='/kaggle/input/EDIT_APPROVED_REAL_IMAGES/facial_real_split_v2'
PASSED_REAL_LABEL_AUDIT='/kaggle/input/EDIT_REAL_LABEL_AUDIT/label_audit_summary.json'
IMAGE_ROOT,real_train_evidence=prepare_lora_train_only(
    real_split_manifest=REAL_TRAIN_MANIFEST,real_image_root=REAL_TRAIN_IMAGE_ROOT,
    cls='Hyperpigmentation',stage_dir='/kaggle/working/approved_Hyperpigmenatation_lora_input',
    origin_root=RAW_REAL_DATASET_V2,label_audit_json=PASSED_REAL_LABEL_AUDIT)
AUTO_SPLIT=False
SPLIT_CSV=''
# The global group-safe split is reused; show this class's training/validation/test counts.
print('Class split (before subtype selection):',real_train_evidence['class_frozen_split_counts'])
from dermsynreal.kaggle_runs import start_notebook_logging
PRIVATE_LORA_CONSOLE=start_notebook_logging(OUT_DIR,kind='lora_Hyperpigmentation',
    config={'class_split_counts':real_train_evidence['class_frozen_split_counts'],
            'train_only_image_count':real_train_evidence['count'],
            'split_sha256':real_train_evidence['manifest_sha256']},repo_root=DERMSYNREAL_REPO)
print('Research console log:',PRIVATE_LORA_CONSOLE)
print('Approved train-only source:',IMAGE_ROOT,'images:',real_train_evidence['count'])
def run_logged(command,path):
    import subprocess
    from pathlib import Path
    log=Path(path);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8',errors='replace') as f:
        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1)
        for line in proc.stdout:
            print(line,end='');f.write(line);f.flush()
        code=proc.wait()
    if code:raise subprocess.CalledProcessError(code,command)


## 3. Read the images and find candidate near-duplicate groups
SHA-256 and flip-aware perceptual hashing are used to form provisional groups, never to assert patient identity. For final research, use your externally audited patient/group-level splits.

In [ ]:
EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
all_images = sorted(p for p in IMAGE_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in EXTS)
if not all_images:
    raise ValueError(f'No JPG/PNG/WebP images inside {IMAGE_ROOT}')

records = []
for fp in all_images:
    try:
        with Image.open(fp) as img:
            norm = ImageOps.exif_transpose(img).convert('RGB')
            norm.load()
            h = imagehash.phash(norm)
            mirror_h = imagehash.phash(ImageOps.mirror(norm))
            rgb_hash = hashlib.sha256(norm.tobytes()).hexdigest()
        records.append({'path': fp, 'rel': fp.relative_to(IMAGE_ROOT).as_posix(),
                        'phash': h, 'mirror_phash': mirror_h, 'pixel_sha256': rgb_hash})
    except (OSError, ValueError) as exc:
        print('Skipping unreadable image:', fp, exc)
assert records, 'All images failed to decode'
print('Usable images:', len(records))

In [ ]:
# Complete-link exact and flip-aware pHash candidate grouping via union-find.
# pHash <=5 is a provisional near-duplicate trigger, NOT a medical or patient identity proof.
parent = list(range(len(records)))
def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(i, j):
    ri, rj = root(i), root(j)
    if ri != rj:
        parent[rj] = ri

for i, a in enumerate(records):
    for j in range(i):
        b = records[j]
        is_exact = a['pixel_sha256'] == b['pixel_sha256']
        is_near = min(a['phash'] - b['phash'],
                      a['phash'] - b['mirror_phash'],
                      a['mirror_phash'] - b['phash'],
                      a['mirror_phash'] - b['mirror_phash']) <= 5
        if is_exact or is_near:
            union(i, j)
for i, r in enumerate(records):
    r['group'] = root(i)
print('Provisional groups:', len({r['group'] for r in records}))

## 4. Enforce split boundaries
The script never uses validation or test images for generator training. If the dataset is already divided into train/val/test, only train/ is read. For ungrouped data, the provisional manifest is written to disk for future review.

In [ ]:
def csv_lookup(csv_file, field):
    if not csv_file:
        return {}
    out = {}
    with open(csv_file, newline='', encoding='utf-8-sig') as fh:
        reader = csv.DictReader(fh)
        if not {'filename', field}.issubset(reader.fieldnames or []):
            raise ValueError(f'{csv_file} must include filename,{field} columns')
        for row in reader:
            key = row['filename'].strip().replace('\\', '/')
            if key in out:
                raise ValueError(f'Duplicate filename in CSV: {key}')
            out[key] = row[field].strip()
    return out

split_lookup = csv_lookup(SPLIT_CSV, 'split')
caption_lookup = csv_lookup(CAPTIONS_CSV, 'caption')
basename_counts = Counter(Path(r['rel']).name for r in records)

def lookup(data, rec):
    if rec['rel'] in data:
        return data[rec['rel']]
    base = Path(rec['rel']).name
    if base in data:
        if basename_counts[base] != 1:
            raise ValueError(f'Ambiguous basename in metadata: {base}; use relative paths')
        return data[base]
    return None

train_path = IMAGE_ROOT / 'train'
if train_path.is_dir():
    for rec in records:
        rec['split'] = 'train' if train_path in rec['path'].parents else 'held_out'
    split_source = 'train/ directory'
elif SPLIT_CSV:
    for rec in records:
        value = lookup(split_lookup, rec)
        rec['split'] = (value or 'unassigned').lower()
    split_source = 'SPLIT_CSV'
elif AUTO_SPLIT:
    if len({r['group'] for r in records}) < 5:
        raise ValueError('Too few independent groups for provisional train/val/test split')
    indices = np.arange(len(records))
    groups = np.array([r['group'] for r in records])
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_ix, rest_ix = next(gss.split(indices, groups=groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED + 1)
    sub_val_ix, sub_test_ix = next(gss2.split(rest_ix, groups=groups[rest_ix]))
    val_ix, test_ix = rest_ix[sub_val_ix], rest_ix[sub_test_ix]
    train_set, val_set = set(train_ix), set(val_ix)
    for i, rec in enumerate(records):
        rec['split'] = ('train' if i in train_set else
                        'val' if i in val_set else 'test')
    split_source = 'AUTO_SPLIT provisional groups'
else:
    raise ValueError('Provide train/ or SPLIT_CSV, or enable AUTO_SPLIT')

# Never train on any group whose members were held out in a user-provided split.
held_groups = {r['group'] for r in records if r['split'] != 'train'}
train_records = [r for r in records if r['split'] == 'train' and r['group'] not in held_groups]
if not train_records:
    raise ValueError('No safe training images selected; review group overlaps and split CSV')

manifest = OUT_DIR / 'split_manifest.csv'
with manifest.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['filename', 'group', 'split'])
    writer.writeheader()
    for r in records:
        writer.writerow({'filename': r['rel'], 'group': r['group'], 'split': r['split']})
print('Split policy:', split_source, '| train images:', len(train_records), '| manifest:', manifest)

## 5. Prepare correctly formatted Hugging Face ImageFolder
`metadata.csv` is at the data root. **Relative paths include the `images/` subdirectory**, which is necessary for ImageFolder to associate captions with files.

**Image-preprocessing safeguard:** convert transparency over a neutral background and pad images to 512×512 (preserve face extent instead of cropping).

In [ ]:
images_dir = DATA_PREP / 'images'
images_dir.mkdir(parents=True, exist_ok=True)
meta = DATA_PREP / 'metadata.csv'
with meta.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['file_name', 'text'])
    writer.writeheader()
    for i, r in enumerate(train_records):
        src = r['path']
        # Normalize all file formats to RGB PNG, avoiding alpha and EXIF inconsistencies.
        name = f'{i:06d}.png'
        dst = images_dir / name
        with Image.open(src) as im:
            canonical = ImageOps.exif_transpose(im)
            # Transparency must not turn into black areas after RGB conversion.
            if canonical.mode in ('RGBA', 'LA') or 'transparency' in canonical.info:
                rgba = canonical.convert('RGBA')
                background = Image.new('RGBA', rgba.size, (215, 215, 215, 255))
                background.alpha_composite(rgba)
                rgb = background.convert('RGB')
            else:
                rgb = canonical.convert('RGB')
            # Pad, rather than crop, rectangular clinical face photographs:
            # avoid removing forehead, cheeks, or chin morphology.
            square = ImageOps.pad(rgb, (RESOLUTION, RESOLUTION),
                                  method=Image.Resampling.LANCZOS,
                                  color=(215, 215, 215), centering=(0.5, 0.5))
            square.save(dst)
        caption = lookup(caption_lookup, r)
        if not caption:
            caption = ('Clinical photograph of a face showing hyperpigmentation with natural skin texture '
                       'and areas of uneven brown pigmentation')
        writer.writerow({'file_name': f'images/{name}', 'text': caption})

# Check metadata schema and physical filenames before expensive model download.
with meta.open(newline='', encoding='utf-8') as fh:
    entries = list(csv.DictReader(fh))
assert len(entries) == len(train_records)
assert all((DATA_PREP / row['file_name']).is_file() and row['text'].strip() for row in entries)
print('Captioned training images:', len(entries), '| metadata:', meta)

## 6. Verify a real `datasets.load_dataset('imagefolder')` read
The next check runs on Kaggle without needing GPU, but requires the installed datasets package.

In [ ]:
from datasets import load_dataset
sample = load_dataset('imagefolder', data_dir=str(DATA_PREP), split='train')
assert {'image', 'text'}.issubset(sample.column_names), sample.column_names
assert len(sample) == len(train_records), (len(sample), len(train_records))
print('ImageFolder successfully read:', len(sample), 'rows; columns:', sample.column_names)
display(sample[0]['image'].resize((256, 256)))
print('First caption:', sample[0]['text'])
# Validate caption length using the SD2.1 training tokenizer (single CLIP chunk).
from transformers import CLIPTokenizer
train_tokenizer = CLIPTokenizer.from_pretrained(BASE_MODEL,subfolder='tokenizer')
for i,caption in enumerate(sample['text']):
    n=len(train_tokenizer(caption,truncation=False,add_special_tokens=False)['input_ids'])
    if n>75:
        raise ValueError(f'Training caption {i} contains {n} content tokens (maximum 75); shorten it.')
print('All training captions fit SD2.1 training context (75 content tokens).')


## 7. Fetch the version-matched official training script
The script is pinned to **Diffusers v0.35.1** and is not automatically replaced with the moving `main` branch. We explicitly verify that important argument flags exist before launching.

In [ ]:
SCRIPT_DIR = Path('/kaggle/working/diffusers_examples')
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT = SCRIPT_DIR / 'train_text_to_image_lora.py'
SCRIPT_URL = ('https://raw.githubusercontent.com/huggingface/diffusers/v0.35.1/'
              'examples/text_to_image/train_text_to_image_lora.py')
if not SCRIPT.exists():
    urllib.request.urlretrieve(SCRIPT_URL, SCRIPT)
script_text = SCRIPT.read_text(encoding='utf-8')
assert 'check_min_version("0.35.0")' in script_text
for arg in ('--train_data_dir', '--mixed_precision', '--use_8bit_adam', '--rank', '--gradient_checkpointing'):
    assert arg in script_text, f'Official script missing expected option {arg}'
print('Trainer script obtained:', SCRIPT, '| bytes:', SCRIPT.stat().st_size)

## 8. Train LoRA on the Kaggle GPU
For T4, use FP16, 8-bit Adam and gradient checkpointing. Validation image generation is **disabled inside training** to reduce VRAM pressure; Notebook 01 evaluates saved weights. Change `STEPS=20` to the intended steps only after a smoke test.

In [ ]:
base_revision = model_info(BASE_MODEL).sha
# Save a machine-readable record of the backbone used to train this LoRA.
provenance = {
    'base_model': BASE_MODEL,
    'base_revision': base_revision,
    'adapter_type': 'custom_unet_lora',
    'published_textual_inversion_used_during_training': False,
    'published_inference_adapters': ['hcarrion/hyperpigmentation', 'hcarrion/solar_lentigo'],
    'training_image_count': len(train_records),
    'resolution': RESOLUTION,
    'lora_rank': RANK,
    'training_steps_requested': STEPS,
    'seed': SEED,
    'note': 'Published textual inversions can be loaded alongside this LoRA at inference.',
}
(OUT_DIR / 'training_provenance.json').write_text(
    json.dumps(provenance, indent=2), encoding='utf-8'
)
command = [
    sys.executable, '-m', 'accelerate.commands.launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    str(SCRIPT),
    '--pretrained_model_name_or_path', BASE_MODEL,
    '--revision', base_revision,
    '--train_data_dir', str(DATA_PREP),
    '--image_column', 'image',
    '--caption_column', 'text',
    '--resolution', str(RESOLUTION),
    '--center_crop',
    '--train_batch_size', str(BATCH_SIZE),
    '--gradient_accumulation_steps', str(GRAD_ACCUM),
    '--max_train_steps', str(STEPS),
    '--learning_rate', str(LR),
    '--lr_scheduler', 'constant',
    '--lr_warmup_steps', '0',
    '--output_dir', str(OUT_DIR),
    '--checkpointing_steps', str(CHECKPOINT_STEPS),
    '--rank', str(RANK),
    '--seed', str(SEED),
    '--mixed_precision', 'fp16',
    '--gradient_checkpointing',
    '--report_to', 'none',
]
if USE_8BIT_ADAM:
    command.append('--use_8bit_adam')
print('Training command:', ' '.join(command))
run_logged(command,OUT_DIR/'trainer_console.log')

## 9. Verify and package trained weights
Do not assume LoRA checkpoint exists if training failed. The final output can be attached to the inference notebook as a Kaggle input.

In [ ]:
weights = OUT_DIR / 'pytorch_lora_weights.safetensors'
if not weights.is_file() or weights.stat().st_size == 0:
    raise RuntimeError(f'Expected LoRA weights missing or empty: {weights}')
print('LoRA weights:', weights, 'bytes:', weights.stat().st_size)
import zipfile
lora_zip = Path('/kaggle/working/hyperpigmentation_lora_weights.zip')
with zipfile.ZipFile(lora_zip, 'w', compression=zipfile.ZIP_STORED) as bundle:
    bundle.write(weights, weights.name)
    provenance_path = OUT_DIR / 'training_provenance.json'
    if provenance_path.is_file():
        bundle.write(provenance_path, provenance_path.name)
print('Download:', lora_zip)

In [ ]:
# Retain the **three class split counts** beside the LoRA weight/checkpoint.
import shutil
from pathlib import Path
for _name in ('class_frozen_split_counts.csv','class_frozen_split_assignments.csv','lora_train_source_summary.json'):
    shutil.copy2(Path(IMAGE_ROOT)/_name,Path(OUT_DIR)/_name)
# Save LoRA checkpoint evidence without claiming it was medically validated.
append_run_log(OUT_DIR,kind='lora_fine_tuning',config={'class':'Hyperpigmentation',
    'base_model':BASE_MODEL,'training_steps':STEPS,'seed':SEED,
    'approved_real_train_count':real_train_evidence['count'],
    'approved_manifest_sha256':real_train_evidence['manifest_sha256']},
    artifacts=[OUT_DIR/'trainer_console.log',
        OUT_DIR/'class_frozen_split_counts.csv',OUT_DIR/'pytorch_lora_weights.safetensors'],repo_root=DERMSYNREAL_REPO)
print('LoRA checkpoint and logs:',OUT_DIR)

## References
- [Diffusers v0.35.1 official LoRA training script](https://raw.githubusercontent.com/huggingface/diffusers/v0.35.1/examples/text_to_image/train_text_to_image_lora.py)
- [Hugging Face ImageFolder captions](https://huggingface.co/docs/datasets/v2.20.0/en/image_dataset)
- [Stable Diffusion 2.1 base community mirror](https://huggingface.co/sd2-community/stable-diffusion-2-1-base)

Kaggle training is still required to validate the complete GPU workflow; morphology quality needs expert review.